In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaSparkBase") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    ) \
    .getOrCreate()

spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9ae0a559-af73-4b95-9a25-b894e2d69a42;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

In [2]:
print("Spark:", spark.version)
print("Scala JVM:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

Spark: 4.1.1
Scala JVM: 2.13.17


In [11]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "ec-kafka:9092") \
    .option("subscribe", "orden-eventos") \
    .option("startingOffsets", "earliest") \
    .load()

In [12]:
df_kafka.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [13]:
df_texto = df_kafka.selectExpr("CAST(value AS STRING) as mensaje")

In [14]:
#.option("checkpointLocation", "/opt/artifacts/checkpoints/ventas_console") \
query_console = df_texto.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()


26/04/16 00:04:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-abb60894-c929-4626-b32a-42cd41d38187. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/16 00:04:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|                  hi|
|             orden 1|
|{"tipoEvento":"or...|
|{"tipoEvento":"or...|
|{"tipoEvento":"or...|
|{"tipoEvento":"or...|
+--------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"tipoEvento":"or...|
+--------------------+



In [32]:
# Convertir JSON
df_value = df_kafka.selectExpr("CAST(value AS STRING)")

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

schema = StructType() \
    .add("tipoEvento", StringType()) \
    .add("ordenId", IntegerType()) \
    .add("total", DoubleType()) \
    .add("estado", StringType()) \
    .add("timestamp", DoubleType())

df_parsed = df_value.select(
    from_json(col("value"), schema).alias("data")
).select("data.*")

In [20]:
df_parsed

DataFrame[tipoEvento: string, ordenId: int, estado: string, timestamp: double]

In [33]:

# Transformación
df_transformed = df_parsed.filter(col("total") > 1)

In [34]:
# Salida consola (debug)
df_transformed.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/16 00:49:07 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-34fb1cd3-0f73-4262-8094-60bff8b58f64. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/16 00:49:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----------+-------+-----+------+---------+
|tipoEvento|ordenId|total|estado|timestamp|
+----------+-------+-----+------+---------+
+----------+-------+-----+------+---------+



-------------------------------------------
Batch: 4
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"tipoEvento":"or...|
+--------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+------------+-------+---------+---------+
|  tipoEvento|ordenId|   estado|timestamp|
+------------+-------+---------+---------+
|orden.creada|      7|PENDIENTE|     NULL|
+------------+-------+---------+---------+

+------------+-------+-----+---------+---------+
|  tipoEvento|ordenId|total|   estado|timestamp|
+------------+-------+-----+---------+---------+
|orden.creada|      7| 90.0|PENDIENTE|     NULL|
+------------+-------+-----+---------+---------+



In [27]:
query_parquet = df_transformed.writeStream \
    .format("parquet") \
    .option("path", "data/orden_eventos_parquet") \
    .option("checkpointLocation", "checkpoint/orden-eventos") \
    .outputMode("append") \
    .start()



26/04/16 00:22:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/16 00:22:25 WARN StreamingQueryManager: Stopping existing streaming query [id=83b9ce13-5cf5-4fab-bb5d-e0476ab05e3a, runId=d94cf194-7ed9-4680-a8cc-90f12b6deb8b], as a new run is being started.
26/04/16 00:22:25 WARN DAGScheduler: Failed to cancel job group d94cf194-7ed9-4680-a8cc-90f12b6deb8b. Cannot find active jobs for it.
26/04/16 00:22:25 WARN DAGScheduler: Failed to cancel job group d94cf194-7ed9-4680-a8cc-90f12b6deb8b. Cannot find active jobs for it.


In [28]:
ls data/orden_eventos_parquet

_spark_metadata/
part-00000-5b81da30-f6ac-48a1-80c9-b26d38dbd608-c000.snappy.parquet


In [30]:
#spark.streams.awaitAnyTermination()
query_parquet.awaitTermination()

In [31]:
df_final = spark.read.parquet("data/orden_eventos_parquet")
df_final.show()

+------------+-------+---------+---------+
|  tipoEvento|ordenId|   estado|timestamp|
+------------+-------+---------+---------+
|orden.creada|      2|PENDIENTE|     NULL|
|orden.creada|      3|PENDIENTE|     NULL|
|orden.creada|      4|PENDIENTE|     NULL|
|orden.creada|      5|PENDIENTE|     NULL|
|orden.creada|      6|PENDIENTE|     NULL|
+------------+-------+---------+---------+

